# SS3 Part B — Fraud Detection for STADIOalot with the SS2 Hybrid Model
**CAP182 Capstone · applying the final method from SS2 to the real client data extract**

**Business problem (SS1):** STADIOalot loses money to fraudulent orders (card fraud, chargebacks). Fraud is rare, so the client needs a model that ranks every new order by fraud risk so analysts can review the riskiest ones first, with few false alarms.

**Final method from SS2 (Part C comparison and Part D recommendation): Model 2, the hybrid Isolation Forest + Neural Network.** It beat the Random Forest (Model 1) on PR-AUC (0.615 vs 0.335) on the SS2 benchmark data.
1. An **Isolation Forest** is fitted on *legitimate* training orders only, and produces three anomaly features: `anomaly_score`, `anomaly_flag`, `anomaly_bin`.
2. A **feed-forward neural network** (64 → 32 → 1, dropout 0.3, Adam, SMOTE 0.5 on training data only) is trained on the original features plus the three anomaly features and outputs the fraud probability.

Part D listed adaptations needed for STADIOalot's data (customer-based velocity features, chargeback + flag labels, contextual features, risk-tiering, leakage-safe preprocessing). They are applied here and flagged where used.

| # | Component | Marks |
|---|---|---|
| 1 | Dataset loading | 2 |
| 2 | Dataset preprocessing (incl. train / validation / test split) | 5 |
| 3 | Model training | 5 |
| 4 | Model validation (unseen test set) | 5 |
| 5 | Results visualisation | 3 |

> **POPIA:** name, email and phone are never loaded; date of birth is used only to derive an age. The data folder is git-ignored.

In [ ]:
import os, json, zipfile, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
                             confusion_matrix, precision_score, recall_score, f1_score)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold'})
SEED = 42
np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
os.makedirs('outputs', exist_ok=True)

I0000 00:00:1792177325.752922    1321 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1792177327.064866    1321 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


---
# 1. DATASET LOADING
Only the tables relevant to fraud are loaded. The other tables (search history, abandoned carts, product listings, seller profiles, complaints) belong to other problems or cannot be linked to an order (`01_orders` has no `seller_id` / `listing_id`); `10_purchase_histories` is a lifetime summary that includes the orders being predicted, so using it would leak the answer.

| Table | Use |
|---|---|
| `01_orders.csv` | One row per order (the unit we score) |
| `02_payments.csv` | Amount, currency, payment method |
| `09_customer_demographics.csv` | Age, tenure, country, city and province only (PII columns are not read) |
| `03_chargebacks.csv`, `04_flagged_transactions.csv` | **Labels only** (post-event) |

**Data format justification:** CSV files read with pandas and joined on `order_id` / `customer_id` into one row per paid order. This matches the SS2 pipeline, which also worked on a flat one-row-per-transaction table.

In [ ]:
ZIP_PATH, DATA_DIR = 'stadioalot_data extract.zip', 'data/stadioalot'
if not os.path.exists(DATA_DIR):
    assert os.path.exists(ZIP_PATH), f'Put {ZIP_PATH} next to this notebook (or extract to {DATA_DIR})'
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall('data')

DEMO_COLS = ['customer_id', 'date_of_birth', 'location_city', 'location_state', 'location_country', 'account_creation_date']  # no PII
orders_raw = pd.read_csv(f'{DATA_DIR}/01_orders.csv')
pay_raw    = pd.read_csv(f'{DATA_DIR}/02_payments.csv')
demo_raw   = pd.read_csv(f'{DATA_DIR}/09_customer_demographics.csv', usecols=DEMO_COLS)
cb_raw     = pd.read_csv(f'{DATA_DIR}/03_chargebacks.csv')
flag_raw   = pd.read_csv(f'{DATA_DIR}/04_flagged_transactions.csv')
for n, d in [('orders', orders_raw), ('payments', pay_raw), ('demographics', demo_raw), ('chargebacks', cb_raw), ('flagged_transactions', flag_raw)]:
    print(f'{n:22s} rows={len(d):6d} cols={d.shape[1]}')

# Target. PRIMARY label 'is_fraud': fraudulent chargeback OR flag resolved as Blocked / Escalated to bank.
# Part D of SS2 also proposed 'flagged OR charged back' -> kept as ALTERNATIVE label 'is_fraud_broad' (sensitivity check in section 4).
fraud_ids = set(cb_raw.loc[cb_raw.chargeback_reason.str.strip().str.lower() == 'fraudulent', 'order_id']) | \
            set(flag_raw.loc[flag_raw.resolution.str.strip().isin(['Blocked', 'Escalated to bank']), 'order_id'])
broad_ids = set(cb_raw.order_id) | set(flag_raw.order_id)
print(f'Primary fraud orders: {len(fraud_ids)} | broad (any flag or chargeback): {len(broad_ids)}')
orders_raw.head()

orders                 rows=  8963 cols=10
payments               rows=  8412 cols=5
demographics           rows=  5218 cols=6
chargebacks            rows=   410 cols=6
flagged_transactions   rows=  1728 cols=7
Primary fraud orders: 401 | broad (any flag or chargeback): 2014


,order_id,customer_id,order_date,order_time,order_status,order_quantities,comments,star_ratings,paid_date,shipped_date
0,12300000,984052,2025-06-14,23:10:22,Pending,1,Ring the bell twice,NaN,2025-06-14 23:13:53,NaN
1,12300001,980180,2025-09-19,09:39:40,Shipped,2,Deliver after 5pm please,NaN,2025-09-19 10:09:45,2025-09-23
2,12300002,983949,2025-05-02,14:52:03,Cancelled,1,"Fragile, handle with care",NaN,NaN,NaN
3,12300003,983635,2025-09-21,06:53:52,Shipped,2,Leave at back door,NaN,2025-09-21 07:23:28,2025-09-22
4,12300004,982899,2025-01-06,22:53:35,Delivered,1,NaN,NaN,2025-01-06 23:23:59,2025-01-09


### Dataset summary
* **Raw extract:** 8,963 order rows (17 exact duplicates) → **8,946 unique orders**, 2025-01-01 to 2026-09-06.
* **Modelling population:** **8,412 paid orders**. The 534 cancelled orders have no payment (no transaction took place) so they are not scored.
* **Classes:** binary, **0 = legitimate**, **1 = confirmed fraud** (fraudulent chargeback, or blocked / escalated-to-bank flag).
* **Counts:** 8,011 legitimate vs **401 fraud**.
* **Imbalance:** fraud is **4.77%** of orders, about **1 fraud per 20 legitimate orders**. This is far less extreme than the SS2 benchmark (0.39%), but still strongly imbalanced, so accuracy is meaningless and SS2's imbalance-aware metrics (PR-AUC, precision, recall) are used.

---
# 2. DATASET PREPROCESSING
Same four feature families as SS2 (temporal, contextual, behavioural velocity, customer history) plus risk-tiering of high-cardinality categoricals. Everything that *learns* from data (risk tiers, scaler, Isolation Forest, SMOTE, network) uses the **training subset only**.

In [ ]:
# ---- 2.1 Cleaning --------------------------------------------------------------------------------
orders = orders_raw.drop_duplicates().copy()
for d in (orders, pay_raw, demo_raw):
    for c in d.select_dtypes('object').columns:
        d[c] = d[c].astype(str).str.strip().replace({'nan': np.nan})
pay = pay_raw.copy()
pay['payment_method'] = pay['payment_method'].str.title().replace({'Cod': 'COD', 'Instant Eft': 'Instant EFT'})
pay['currency_code']  = pay['currency_code'].str.upper()
orders['order_dt'] = pd.to_datetime(orders['order_date'] + ' ' + orders['order_time'], errors='coerce')
orders['paid_dt']  = pd.to_datetime(orders['paid_date'], errors='coerce')
demo = demo_raw.copy()
demo['dob'] = pd.to_datetime(demo['date_of_birth'], errors='coerce'); demo['acct_created'] = pd.to_datetime(demo['account_creation_date'], errors='coerce')

# ---- 2.2 Currency -> ZAR (SS2 Part D: 'confirm all amounts are in ZAR'). Fixed approximate rates (assumption) ---
ZAR_RATE = {'ZAR': 1.0, 'USD': 18.0, 'EUR': 20.0, 'GBP': 23.0}
pay['amount_zar'] = pay['payment_amount'] * pay['currency_code'].map(ZAR_RATE)

# ---- 2.3 Join, labels, range checks -----------------------------------------------------------------
df = (orders.merge(pay[['order_id', 'payment_method', 'currency_code', 'amount_zar']], on='order_id', how='inner')
            .merge(demo[['customer_id', 'dob', 'acct_created', 'location_city', 'location_state', 'location_country']], on='customer_id', how='left'))
df['is_fraud'] = df.order_id.isin(fraud_ids).astype(int)
df['is_fraud_broad'] = df.order_id.isin(broad_ids).astype(int)
n0 = len(df); df = df[(df.amount_zar > 0) & (df.order_quantities > 0) & df.order_dt.notna()]
print(f'Dropped {n0-len(df)} rows failing range/format checks -> {len(df):,} paid orders')

Dropped 0 rows failing range/format checks -> 8,412 paid orders


In [ ]:
# ---- 2.4 Feature engineering (all point-in-time: only information known when the order was placed) ----
df = df.sort_values('order_dt').reset_index(drop=True)
# Temporal
df['trans_hour'] = df.order_dt.dt.hour; df['trans_dayofweek'] = df.order_dt.dt.dayofweek
df['is_weekend'] = (df.trans_dayofweek >= 5).astype(float); df['is_night'] = (df.trans_hour < 6).astype(float)
# Contextual
df['amt_log'] = np.log1p(df.amount_zar); df['unit_price_log'] = np.log1p(df.amount_zar / df.order_quantities)
df['age_years'] = ((df.order_dt - df.dob).dt.days / 365.25).clip(18, 100)
df['acct_tenure_days'] = (df.order_dt - df.acct_created).dt.days.clip(lower=0)
df['pay_delay_min'] = ((df.paid_dt - df.order_dt).dt.total_seconds() / 60).clip(lower=0)
df['is_foreign_ccy'] = (df.currency_code != 'ZAR').astype(float)
df['is_foreign_cust'] = (df.location_country.fillna('South Africa') != 'South Africa').astype(float)
df['has_comment'] = df.comments.notna().astype(float)          # SS2 Part D: use order comments (free text is only a delivery note, so a presence flag)
# Behavioural velocity per CUSTOMER (SS2 Part D: group by customer_id instead of card number). Backward-looking only.
def velocity(g):
    t = g.order_dt.values.astype('datetime64[s]').astype('int64'); a = g.amount_zar.values
    cnt, tot, gap = np.zeros(len(g)), np.zeros(len(g)), np.full(len(g), np.nan)
    for i in range(len(g)):
        m = (t[:i] >= t[i] - 86400)
        cnt[i], tot[i] = m.sum(), a[:i][m].sum()
        if i: gap[i] = (t[i] - t[i-1]) / 60
    return pd.DataFrame({'txn_count_24h': cnt, 'txn_amt_sum_24h': tot, 'mins_since_last_txn': gap}, index=g.index)
df = df.join(df.groupby('customer_id', group_keys=False).apply(velocity))
df['is_first_order'] = df.mins_since_last_txn.isna().astype(float)
# Customer history: z-score of this amount vs the customer's own PRIOR orders (shifted so the order itself is excluded)
g = df.groupby('customer_id')['amount_zar']
prior_n = g.cumcount(); prior_mean = (g.cumsum() - df.amount_zar) / prior_n.replace(0, np.nan)
prior_sq = (df.amount_zar**2).groupby(df.customer_id).cumsum() - df.amount_zar**2
prior_std = np.sqrt((prior_sq / prior_n.replace(0, np.nan) - prior_mean**2).clip(lower=0))
df['prior_orders'] = prior_n.astype(float)
df['cust_amt_zscore'] = ((df.amount_zar - prior_mean) / prior_std.replace(0, np.nan)).clip(-10, 10)
pm = pd.get_dummies(df.payment_method, prefix='pay').astype(float); df = pd.concat([df, pm], axis=1)
print('Engineered; example rows:'); df[['txn_count_24h', 'txn_amt_sum_24h', 'mins_since_last_txn', 'cust_amt_zscore', 'prior_orders']].describe().round(2).loc[['mean', 'max']]

Engineered; example rows:


,txn_count_24h,txn_amt_sum_24h,mins_since_last_txn,cust_amt_zscore,prior_orders
mean,0.0,5.73,213122.70,1.71,0.81
max,1.0,8489.76,867314.97,10.00,7.00


In [ ]:
# ---- 2.5 Chronological 60 / 20 / 20 split (train = past, validation = next, test = most recent) ------------
n = len(df); i1, i2 = int(.6 * n), int(.8 * n)
train, val, test = df.iloc[:i1].copy(), df.iloc[i1:i2].copy(), df.iloc[i2:].copy()

# ---- 2.6 Risk-tier grouping of high-cardinality categoricals (SS2 / Carneiro 2016), fitted on TRAIN only ----
def fit_risk_map(col, label, min_n=15, tiers=4):
    st = train.groupby(col)[label].agg(['mean', 'count']); st = st[st['count'] >= min_n]
    return pd.qcut(st['mean'].rank(method='first'), tiers, labels=False).add(1).to_dict()
def add_risk(part, maps):
    for col, mp in maps.items(): part[col + '_risk'] = part[col].map(mp).fillna(2.5)       # unseen / rare -> neutral tier
    return part
NUM = ['trans_hour', 'trans_dayofweek', 'is_weekend', 'is_night', 'amt_log', 'unit_price_log', 'order_quantities', 'age_years',
       'acct_tenure_days', 'pay_delay_min', 'is_foreign_ccy', 'is_foreign_cust', 'has_comment', 'txn_count_24h', 'txn_amt_sum_24h',
       'mins_since_last_txn', 'is_first_order', 'prior_orders', 'cust_amt_zscore']
BASE_FEATURES = NUM + ['location_city_risk', 'location_state_risk'] + list(pm.columns)

def make_matrices(label):
    # Everything learned (risk tiers, medians, scaler) is fitted on train only and re-used on validation and test.
    maps = {c: fit_risk_map(c, label) for c in ['location_city', 'location_state']}
    parts = [add_risk(p.copy(), maps) for p in (train, val, test)]
    med = parts[0][BASE_FEATURES].median()
    sc = StandardScaler().fit(parts[0][BASE_FEATURES].fillna(med))
    X = [sc.transform(p[BASE_FEATURES].fillna(med)) for p in parts]
    y = [p[label].values for p in parts]
    return X, y, dict(maps=maps, medians=med, scaler=sc)

(X_train, X_val, X_test), (y_train, y_val, y_test), prep_artifacts = make_matrices('is_fraud')
split_tbl = pd.DataFrame({'orders': [len(train), len(val), len(test)], 'fraud': [y_train.sum(), y_val.sum(), y_test.sum()],
                          'fraud_%': [100*y_train.mean(), 100*y_val.mean(), 100*y_test.mean()],
                          'from': [p.order_dt.min().date() for p in (train, val, test)], 'to': [p.order_dt.max().date() for p in (train, val, test)]},
                         index=['train', 'validation', 'test']).round(2)
print(len(BASE_FEATURES), 'base features'); split_tbl

29 base features


,orders,fraud,fraud_%,from,to
train,5047,239,4.74,2025-01-01,2026-01-24
validation,1682,74,4.40,2026-01-24,2026-05-16
test,1683,73,4.34,2026-05-17,2026-09-06


### Why this preprocessing?
| Step | Reason |
|---|---|
| **De-duplicate, strip / standardise text** | The client README warns the extract is uncleaned; duplicates distort customer history and velocity. |
| **Drop cancelled orders** | No payment means no transaction to score. |
| **Convert to ZAR** | Amounts are in each row's own currency. SS2 Part D required resolving this *before* modelling, otherwise an unconverted 120 GBP looks tiny next to 2,000 ZAR. |
| **log-transform amounts** | Right-skewed order values (same as SS2's `amt_log`). |
| **Temporal features** (hour, weekday, weekend, night) | SS2 temporal family; fraud often clusters at odd hours. |
| **Customer velocity** (orders / spend in trailing 24h, minutes since last order) | SS2's second most important family in Al-Hajabed (2026). Part D said to regroup by `customer_id` instead of card number. Backward-looking only, so no future leakage. |
| **Customer z-score vs own history** | Fraud is often a break from the customer's own pattern; uses earlier orders only. The lifetime `10_purchase_histories` table is *not* used because it contains the order being predicted. |
| **Risk tiers for city and province** | SS2 used train-only fraud-rate tiers (Carneiro, 2016) instead of hundreds of one-hots. Unseen or rare places get a neutral tier. |
| **`has_comment` flag** | SS2 Part D proposed using Comments. These are delivery notes (e.g. "Ring the bell twice"), not reviews, so sentiment is meaningless and only presence is kept. |
| **`star_ratings` excluded** | Part D proposed Star Ratings, but the client README shows they are given *after* delivery, so they would leak the outcome and would not exist at scoring time. |
| **Excluded outcome tables / fields** (flags, chargebacks, resolutions, delivery, shipped date) | All post-event; using them as features would let the model see the answer. |
| **Drop name / email / phone, derive age from DOB** | POPIA data minimisation. |
| **Chronological 60/20/20 split** | SS2 used the benchmark's own train/test files; here the model must predict the *future*, so time order is used. A validation set is added for early stopping and threshold selection so the test set is touched once. |
| **Imputation, risk tiers, scaler fitted on train only** | Follows SS2's leakage-prevention rule. |
| **SMOTE only inside training (section 3)** | As in SS2, validation and test keep their natural class balance so metrics are realistic. |

---
# 3. MODEL TRAINING
**Stage 1: Isolation Forest** fitted on legitimate training orders only. **Stage 2: neural network** trained on base features + `anomaly_score`, `anomaly_flag`, `anomaly_bin` after SMOTE on the training set.

### Hyper-parameters: SS2 value, value used here, and why
| Parameter | SS2 | Used here | Justification |
|---|---|---|---|
| IF `n_estimators` | 200 | **200** | Unchanged: SS2 setting, plenty of trees for stable scores. |
| IF `contamination` | "auto" | **"auto"** | Unchanged. The fraud rate is not assumed; `anomaly_flag` uses the original-paper score threshold. |
| IF training data | legit only | **legit only** | Unchanged: the forest learns what "normal" looks like. |
| `anomaly_bin` | 5 equal-frequency bins | **5**, edges from train scores | Unchanged; test scores use the *train* edges. |
| SMOTE `sampling_strategy` | 0.5 | **0.5** (train only) | Unchanged: fraud becomes about a third of the training set. |
| Layers | 64 → 32 → 1 (ReLU, sigmoid) | **same** | Unchanged architecture. |
| Dropout | 0.3 | **0.3** | Unchanged: regularisation, even more important with only about 239 real training frauds. |
| Optimiser / learning rate | Adam, 0.001 | **Adam, 0.001** | Unchanged: robust default for small dense networks. |
| Loss | binary cross-entropy | **binary cross-entropy** | Unchanged: probabilistic output for ranking and thresholds. |
| Batch size | 2048 | **128** | *Adapted.* SS2 had 1.3 million rows; here there are about 5,047. At 2048 an epoch would be only a few gradient steps, so 128 keeps a comparable number of updates per epoch. |
| Epochs | 5 fixed | **up to 60 with early stopping** (patience 8, best weights restored, monitor *validation PR-AUC*) | *Adapted, and exactly SS2 Part D's own recommendation:* "train longer with early stopping monitoring validation PR-AUC". 5 fixed epochs is far too few steps on a dataset this small. PR-AUC is used because fraud is rare. |
| Seeds | 42 | **42** | Reproducibility. |

In [ ]:
# ============ Stage 1: Isolation Forest (fit on legitimate TRAIN orders only) ============
def stage1(X_tr, y_tr, Xs):
    iso = IsolationForest(n_estimators=200, contamination='auto', random_state=SEED, n_jobs=-1).fit(X_tr[y_tr == 0])
    score = lambda X: -iso.score_samples(X)                       # higher = more anomalous
    s_tr = score(X_tr)
    edges = np.quantile(s_tr, [.2, .4, .6, .8])                   # 5 equal-frequency bins from TRAIN scores
    def feats(X):
        s = score(X)
        return np.column_stack([s, (iso.predict(X) == -1).astype(float), np.digitize(s, edges).astype(float)])
    return iso, edges, feats

# ============ Stage 2: neural network on base + anomaly features ============
def build_nn(n_in):
    m = keras.Sequential([keras.layers.Input((n_in,)), keras.layers.Dense(64, activation='relu'), keras.layers.Dropout(.3),
                          keras.layers.Dense(32, activation='relu'), keras.layers.Dropout(.3), keras.layers.Dense(1, activation='sigmoid')])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=[keras.metrics.AUC(curve='PR', name='pr_auc'), keras.metrics.AUC(name='roc_auc')])
    return m

def run_hybrid(Xs, ys, use_anomaly=True, epochs=60, verbose=0):
    X_tr, X_va, X_te = Xs; y_tr, y_va, y_te = ys
    iso = edges = None
    if use_anomaly:
        iso, edges, feats = stage1(X_tr, y_tr, Xs)
        X_tr, X_va, X_te = (np.hstack([X, feats(X)]) for X in (X_tr, X_va, X_te))
    X_sm, y_sm = SMOTE(sampling_strategy=0.5, random_state=SEED).fit_resample(X_tr, y_tr)        # TRAIN only
    tf.keras.utils.set_random_seed(SEED)
    nn = build_nn(X_tr.shape[1])
    es = keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=8, restore_best_weights=True)
    hist = nn.fit(X_sm, y_sm, validation_data=(X_va, y_va), epochs=epochs, batch_size=128, callbacks=[es], verbose=verbose)
    p = [nn.predict(X, verbose=0).ravel() for X in (X_tr, X_va, X_te)]
    return dict(iso=iso, edges=edges, nn=nn, hist=hist.history, p_train=p[0], p_val=p[1], p_test=p[2],
                F=(X_tr, X_va, X_te), n_smote=(len(y_sm), int(y_sm.sum())))

t0 = time.time()
H = run_hybrid((X_train, X_val, X_test), (y_train, y_val, y_test), use_anomaly=True, verbose=0)
hist_df = pd.DataFrame(H['hist']); hist_df.index.name = 'epoch'
print(f"Trained in {time.time()-t0:.0f}s | epochs run: {len(hist_df)} | best epoch: {hist_df.val_pr_auc.idxmax()+1} | SMOTE train set: {H['n_smote'][0]:,} rows ({H['n_smote'][1]:,} fraud)")

# ---- capture training outputs for later analysis ----
hist_df.to_csv('outputs/training_history.csv')
H['nn'].save('outputs/model_neural_network.keras'); joblib.dump({'iso': H['iso'], 'edges': H['edges'], **prep_artifacts, 'features': BASE_FEATURES}, 'outputs/model_isolation_forest_and_prep.joblib')
json.dump({'if': {'n_estimators': 200, 'contamination': 'auto', 'fit_on': 'legit train only'}, 'smote_ratio': 0.5,
           'nn': {'layers': [64, 32, 1], 'dropout': .3, 'optimizer': 'Adam', 'lr': 1e-3, 'batch_size': 128, 'max_epochs': 60, 'early_stopping': 'val_pr_auc, patience 8'},
           'epochs_run': len(hist_df), 'best_epoch': int(hist_df.val_pr_auc.idxmax()) + 1, 'seed': SEED}, open('outputs/training_config.json', 'w'), indent=2)
hist_df.round(4).tail(4)

Trained in 5s | epochs run: 9 | best epoch: 1 | SMOTE train set: 7,212 rows (2,404 fraud)


,loss,pr_auc,roc_auc,val_loss,val_pr_auc,val_roc_auc
epoch,,,,,,
5,0.5449,0.5792,0.7569,0.4034,0.0372,0.4381
6,0.5306,0.6002,0.7768,0.3807,0.0369,0.4409
7,0.5117,0.6332,0.7960,0.3728,0.0374,0.4474
8,0.5002,0.6393,0.8077,0.3616,0.0378,0.4552


In [ ]:
print(f'TRAIN      PR-AUC {average_precision_score(y_train, H["p_train"]):.3f}  ROC-AUC {roc_auc_score(y_train, H["p_train"]):.3f}  (base rate {y_train.mean():.3f})')
print(f'VALIDATION PR-AUC {average_precision_score(y_val,   H["p_val"]):.3f}  ROC-AUC {roc_auc_score(y_val,   H["p_val"]):.3f}  (base rate {y_val.mean():.3f})')

TRAIN      PR-AUC 0.075  ROC-AUC 0.618  (base rate 0.047)
VALIDATION PR-AUC 0.039  ROC-AUC 0.455  (base rate 0.044)


---
# 4. MODEL VALIDATION (unseen test set)
The **test set** (the most recent 1,683 orders) was not used for risk tiers, scaling, the Isolation Forest, SMOTE, early stopping or threshold choice. It passes through the *same* fitted pipeline: scaler → Isolation Forest features → network.

**Metrics (as in SS2):** PR-AUC and ROC-AUC (threshold-free), then precision / recall / F1 at (a) the default 0.5 threshold, (b) the **F1-optimal threshold chosen on validation** (SS2 Part D) and (c) a **review-capacity threshold**, flagging the riskiest 10% of orders (cut-off set on validation), which is how analysts would really use the score. Accuracy is not used: always predicting "legitimate" scores 95.7%. For the business view: precision@k (hit-rate in the top-k% reviewed), lift over random review, and rand value of fraud caught. A 200-resample bootstrap gives 95% confidence intervals, because the test set has few frauds.

**Comparisons from SS2 re-run on this data:** (i) the network *without* the anomaly features, to test SS2's claim that the hybrid beats a plain network; (ii) a Random Forest as the SS2 Model 1 comparator; (iii) Isolation Forest scores alone.

In [ ]:
def metrics_at(y, p, thr):
    yh = (p >= thr).astype(int)
    return dict(threshold=float(thr), precision=precision_score(y, yh, zero_division=0), recall=recall_score(y, yh, zero_division=0),
                f1=f1_score(y, yh, zero_division=0), flagged_pct=100 * yh.mean())
def best_f1_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p); f = 2 * pr * rc / np.clip(pr + rc, 1e-9, None)
    return th[np.nanargmax(f[:-1])]
def boot_ci(y, p, fn, n=200, seed=0):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum(): out.append(fn(y[i], p[i]))
    return np.percentile(out, [2.5, 97.5])

p_test, p_val = H['p_test'], H['p_val']
thr_f1 = best_f1_threshold(y_val, p_val)                   # chosen on VALIDATION only
REVIEW_PCT = 10
thr_cap = np.percentile(p_val, 100 - REVIEW_PCT)         # analyst-capacity threshold: flag the riskiest 10%, chosen on VALIDATION
m05, mf1, mcap = metrics_at(y_test, p_test, .5), metrics_at(y_test, p_test, thr_f1), metrics_at(y_test, p_test, thr_cap)
order = np.argsort(-p_test)
pk = {f'top_{k}%': float(y_test[order[:max(1, int(len(y_test) * k / 100))]].mean()) for k in (1, 5, 10, 20)}
res = {'test_orders': len(y_test), 'test_fraud': int(y_test.sum()), 'base_rate': float(y_test.mean()),
       'roc_auc': roc_auc_score(y_test, p_test), 'pr_auc': average_precision_score(y_test, p_test),
       'roc_auc_ci95': boot_ci(y_test, p_test, roc_auc_score).tolist(), 'pr_auc_ci95': boot_ci(y_test, p_test, average_precision_score).tolist(),
       'at_0.5': m05, 'at_val_f1_optimal': mf1, 'at_review_capacity_10pct': mcap, 'precision_at_k': pk,
       'val_roc_auc': roc_auc_score(y_val, p_val), 'val_pr_auc': average_precision_score(y_val, p_val)}
res['lift_at_capacity'] = mcap['precision'] / res['base_rate']
yh = (p_test >= thr_cap).astype(int)
res['fraud_zar_total'] = float(test.loc[y_test == 1, 'amount_zar'].sum()); res['fraud_zar_caught'] = float(test.loc[(y_test == 1) & (yh == 1), 'amount_zar'].sum())
cm = confusion_matrix(y_test, yh)
json.dump(res, open('outputs/test_metrics.json', 'w'), indent=2, default=float)
pd.DataFrame({'order_id': test.order_id.values, 'probability': p_test, 'flagged_at_capacity_threshold': yh, 'is_fraud': y_test}).to_csv('outputs/test_predictions.csv', index=False)
pd.DataFrame({'order_id': val.order_id.values, 'probability': p_val, 'is_fraud': y_val}).to_csv('outputs/validation_predictions.csv', index=False)

show = pd.DataFrame({'ROC-AUC': [res['roc_auc']], 'ROC-AUC 95% CI': [str(np.round(res['roc_auc_ci95'], 3))], 'PR-AUC': [res['pr_auc']],
                     'PR-AUC 95% CI': [str(np.round(res['pr_auc_ci95'], 3))], 'random PR-AUC': [res['base_rate']]}, index=['Hybrid IF+NN (test)']).round(3)
display(show)
display(pd.DataFrame({'threshold 0.5 (SS2 default)': m05, 'validation F1-optimal': mf1, f'review capacity: top {REVIEW_PCT}%': mcap}).T.round(3))
print('Precision@k (test):', {k: round(v, 3) for k, v in pk.items()}, '| lift at capacity threshold:', round(res['lift_at_capacity'], 2))
print(f"Fraud value caught: R{res['fraud_zar_caught']:,.0f} of R{res['fraud_zar_total']:,.0f}"); cm

,ROC-AUC,ROC-AUC 95% CI,PR-AUC,PR-AUC 95% CI,random PR-AUC
Hybrid IF+NN (test),0.513,[0.434 0.591],0.048,[0.036 0.069],0.043


,threshold,precision,recall,f1,flagged_pct
threshold 0.5 (SS2 default),0.500,0.000,0.000,0.000,0.416
validation F1-optimal,0.215,0.043,0.890,0.083,89.067
review capacity: top 10%,0.415,0.060,0.096,0.074,6.952


Precision@k (test): {'top_1%': 0.0, 'top_5%': 0.06, 'top_10%': 0.06, 'top_20%': 0.042} | lift at capacity threshold: 1.38
Fraud value caught: R35,660 of R186,513


array([[1500,  110],
       [  66,    7]])

In [ ]:
# ---- Comparisons re-run from SS2 on the client data (all trained on TRAIN, selected on VALIDATION, reported on TEST) ----
H_nn = run_hybrid((X_train, X_val, X_test), (y_train, y_val, y_test), use_anomaly=False)                    # NN alone (no anomaly features)
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight='balanced_subsample', random_state=SEED, n_jobs=-1).fit(X_train, y_train)
iso_only = -IsolationForest(n_estimators=200, contamination='auto', random_state=SEED, n_jobs=-1).fit(X_train[y_train == 0]).score_samples(X_test)
models = {'Hybrid IF + NN (SS2 Model 2)': p_test, 'NN alone (no anomaly features)': H_nn['p_test'],
          'Random Forest (SS2 Model 1 style)': rf.predict_proba(X_test)[:, 1], 'Isolation Forest scores only': iso_only}
cmp = pd.DataFrame({k: {'ROC-AUC': roc_auc_score(y_test, v), 'PR-AUC': average_precision_score(y_test, v),
                        'PR-AUC 95% CI low': boot_ci(y_test, v, average_precision_score)[0], 'PR-AUC 95% CI high': boot_ci(y_test, v, average_precision_score)[1]}
                    for k, v in models.items()}).T
cmp['random baseline'] = y_test.mean(); cmp.round(3).to_csv('outputs/model_comparison.csv'); display(cmp.round(3))

# ---- Label sensitivity: SS2 Part D's alternative label ('flagged OR charged back') on the same pipeline ----
(Xa_tr, Xa_va, Xa_te), (ya_tr, ya_va, ya_te), _ = make_matrices('is_fraud_broad')
Ha = run_hybrid((Xa_tr, Xa_va, Xa_te), (ya_tr, ya_va, ya_te))
alt = {'label': 'flagged OR charged back', 'fraud_rate_test': float(ya_te.mean()), 'roc_auc': roc_auc_score(ya_te, Ha['p_test']), 'pr_auc': average_precision_score(ya_te, Ha['p_test'])}
json.dump(alt, open('outputs/label_sensitivity.json', 'w'), indent=2); print('Alternative label:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in alt.items()})

,ROC-AUC,PR-AUC,PR-AUC 95% CI low,PR-AUC 95% CI high,random baseline
Hybrid IF + NN (SS2 Model 2),0.513,0.048,0.036,0.069,0.043
NN alone (no anomaly features),0.481,0.045,0.033,0.064,0.043
Random Forest (SS2 Model 1 style),0.504,0.047,0.034,0.069,0.043
Isolation Forest scores only,0.496,0.043,0.032,0.061,0.043


Alternative label: {'label': 'flagged OR charged back', 'fraud_rate_test': 0.236, 'roc_auc': 0.526, 'pr_auc': 0.255}


### Interpretation (tied back to the business problem)
* **Headline: on this client data the SS2 hybrid model does not beat random guessing.** On the unseen test set ROC-AUC is **0.513** (95% CI 0.43–0.59) and PR-AUC is **0.048** (95% CI 0.036–0.069) against a random baseline of 0.043. Both confidence intervals contain the random level, so the result is statistically indistinguishable from chance. On the SS2 benchmark the same architecture reached PR-AUC 0.615 against a 0.4% base rate.
* **The training curves explain it.** Training PR-AUC climbs steadily (the network memorises the SMOTE-generated frauds) while validation PR-AUC never rises above the 0.044 base rate; the best validation epoch was epoch 1, so early stopping restored near-untrained weights. Validation ROC-AUC is 0.455, below 0.5, which is noise and not a real inverse signal.
* **Operating points:** at SS2's default 0.5 threshold the model flags only 0.4% of orders and catches no fraud. The validation F1-optimal threshold is degenerate (it flags 89% of orders, recall 0.89 but precision 4.3%, which is what flagging everything gives). At a realistic analyst capacity (riskiest 10% by validation cut-off, which flagged 7% of test orders) precision is 6.0% (lift 1.4× over random review) and recall 9.6%, with about R36k of the R187k fraud value caught. With only 73 test frauds these figures are well within chance variation.
* **SS2's model ranking does not carry over.** Hybrid PR-AUC 0.048, Random Forest 0.047, NN alone 0.045 and Isolation Forest scores alone 0.043: the direction (hybrid highest) matches SS2, but the differences are tiny and all intervals overlap each other and the random baseline, so the hybrid's advantage cannot be confirmed here.
* **The alternative label from SS2 Part D** (any flag *or* chargeback, 23.6% of test orders) gives ROC-AUC 0.526 and PR-AUC 0.255 against a 0.236 base rate: the same near-random picture, so the problem is not the label choice.
* **Business meaning:** with the current extract the model cannot cut analyst workload or fraud losses and should not be deployed for auto-blocking or prioritisation.